# LangGraph G10 — Parallel work
"Is S001 eligible to register for CS201?" needs three independent checks: attendance,
credits, prerequisites. Running them one after another wastes time. A graph can **fan out**
from one node to several and **fan in** again:

```text
              +-> attendance_check --+
START -> load +-> credit_check ------+-> verdict -> END
              +-> prerequisite_check +
```

Two mechanics make this safe. Nodes that run at the same time write to the same key, so that key
needs a **reducer** (here `operator.add` on a list of findings). And when the number of branches
is only known at run time (one per course in the question), `Send` creates them dynamically.
This is the graph form of a general agentic rule: independent work should not wait in line.

### Step 1 — Static fan-out with a reducer on the shared key

In [ ]:
class EligibilityState(TypedDict, total=False):            # ours
    student_id: str
    course_code: str
    findings: Annotated[list, operator.add]                # LangGraph reducer: concurrent writes are concatenated
    verdict: str

def attendance_check(state):                               # ours: three independent checks
    a = STUDENTS[state["student_id"]]["attendance"]
    return {"findings": [f"attendance {a}% {'OK' if a >= 75 else 'BELOW 75%'}"]}

def credit_check(state):
    total = STUDENTS[state["student_id"]]["credits"] + COURSES[state["course_code"]]["credits"]
    return {"findings": [f"credits after registration {total} {'OK' if total <= 24 else 'OVER CAP'}"]}

def prerequisite_check(state):
    missing = COURSES[state["course_code"]]["prerequisites"]
    return {"findings": [f"prerequisites {missing or 'none'} {'(assumed complete)' if missing else 'OK'}"]}

def verdict(state):                                        # ours: fan-in
    ok = all(("OK" in f or "assumed" in f) for f in state["findings"])
    return {"verdict": ("ELIGIBLE" if ok else "NOT ELIGIBLE") + " - " + "; ".join(state["findings"])}

g = StateGraph(EligibilityState)
for name, fn in [("attendance_check", attendance_check), ("credit_check", credit_check), ("prerequisite_check", prerequisite_check), ("verdict", verdict)]:
    g.add_node(name, fn)
for name in ("attendance_check", "credit_check", "prerequisite_check"):
    g.add_edge(START, name)                                # LangGraph: three edges from START = three parallel branches
    g.add_edge(name, "verdict")                            # LangGraph: verdict waits for all three
g.add_edge("verdict", END)
eligibility = g.compile()

print("S003/CS101 :", eligibility.invoke({"student_id": "S003", "course_code": "CS101", "findings": []})["verdict"])
print("S001/CS201 :", eligibility.invoke({"student_id": "S001", "course_code": "CS201", "findings": []})["verdict"])
print("S002/MA110 :", eligibility.invoke({"student_id": "S002", "course_code": "MA110", "findings": []})["verdict"])

> **What just happened**
>
> START has three outgoing edges, so `attendance_check`, `credit_check` and `prerequisite_check` ran in the same step. Each returned a one-item `findings` list; the operator.add reducer concatenated the three instead of keeping only the last. `verdict` has edges from all three, so it waited for all of them and then ran once, which is why every verdict line lists three findings.

### Step 2 — Dynamic fan-out with `Send`

The routing function returns one `Send` per course mentioned; each carries its own private
state to the worker node. The workers run concurrently and their findings merge through the reducer.

In [ ]:
from langgraph.types import Send                           # LangGraph: dynamically create parallel branches

class MultiState(TypedDict, total=False):                  # ours
    question: str
    student_id: str
    findings: Annotated[list, operator.add]

def fan_out(state: MultiState):                            # ours: one Send per course code in the question
    return [Send("check_course", {"student_id": state["student_id"], "course_code": code, "findings": []})
            for code in re.findall(r"\b[A-Z]{2}\d{3}\b", state["question"])]

def check_course(state: EligibilityState):                 # ours: reuse the whole eligibility graph as the worker
    result = eligibility.invoke(state)                     # LangGraph: a compiled graph called from inside a node
    return {"findings": [f"{state['course_code']}: {result['verdict']}"]}

g = StateGraph(MultiState)
g.add_node("check_course", check_course)
g.add_conditional_edges(START, fan_out, ["check_course"])  # LangGraph: the list of Send objects decides how many workers run
g.add_edge("check_course", END)
multi = g.compile()

out = multi.invoke({"question": "Can S002 take CS101, EE150 and MA110 next semester?", "student_id": "S002", "findings": []})
for finding in out["findings"]:
    print(" -", finding[:110])

> **What just happened**
>
> `fan_out` returned three Send objects, one per course code in the question, so LangGraph created three `check_course` branches at run time, each with its own small input state. Each branch ran the whole eligibility graph from the previous cell and returned one finding; the reducer merged them. Ask about two courses and there would be two branches.

### Recap

- **The problem we started with:** independent checks ran one after another.
- **What we added:** fan-out and fan-in edges, a reducer on the shared key, and `Send` for a run-time number of branches.
- **What you saw in the output:** three checks merged into one verdict; three courses were checked by three concurrent workers.
- **Carry forward:** G11 stops assuming every tool lives in this notebook and pulls tools from other processes.